# Problem 25: Nested Tool Calls

**Difficulty:** Medium | **Framework:** LangGraph

**Categories:** tool-calling, orchestration

This notebook accompanies [Agent Foundry](https://agent-foundry-pi.vercel.app/problems/nested-tool-calls).

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain langchain-openai langchain-community

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must call find_user first to get the user ID
- The user ID from find_user must be passed to get_orders
- The agent must not hallucinate or hardcode user IDs
- No changes to the tool implementations


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def find_user(name: str) -> str:
    """Find a user by name and return their user ID."""
    users = {"alice": "USR-001", "bob": "USR-002", "charlie": "USR-003"}
    user_id = users.get(name.lower())
    if user_id:
        return f"Found user '{name}' with ID: {user_id}"
    return f"No user found with name '{name}'"

@tool
def get_orders(user_id: str) -> str:
    """Get recent orders for a user by their user ID."""
    orders = {
        "USR-001": "Order #1234: Laptop ($999), Order #1235: Mouse ($29)",
        "USR-002": "Order #1236: Keyboard ($79), Order #1237: Monitor ($449)",
        "USR-003": "No recent orders",
    }
    return orders.get(user_id, f"No orders found for user ID: {user_id}")

# BUG: The agent fails to chain find_user -> get_orders properly
# It either hallucinates a user ID or passes the name directly to get_orders
# TODO: Ensure the agent extracts the user ID from find_user and passes it to get_orders
agent = create_react_agent(llm, [find_user, get_orders])

result = agent.invoke({"messages": [("human", "Show me Alice's recent orders")]})
print(result["messages"][-1].content)

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Update the system prompt to explicitly instruct the agent to first look up the user, then use the returned ID for subsequent calls
2. Make sure the agent extracts just the ID from the first tool's output rather than passing the entire response string
3. In LangGraph, you can model this as two sequential nodes where the first node's output feeds into the second


## 7. Evaluation Criteria

Check your solution against these criteria:

- find_user is called first
- Correct ID is passed to get_orders
- No hallucinated IDs
